# ROL 2: Analista de Calidad - Validacion y Deteccion de Patrones
## Caso SUNBURST - Analisis de Gestion de Datos

**Universidad de San Buenaventura** | Gestion de Datos | 3er Semestre

---

### Objetivo
Validar la calidad de los datos generados por el Rol 1 y detectar patrones anomalos. **Analisis critico**: los datos contienen anomalias reales que deben ser detectadas.

### Contenido
1. Marco DAMA DMBOK
2. Carga de Datos
3. Validacion Exhaustiva (nulos, duplicados, FK, fechas, categorias)
4. Metricas DAMA (Completitud, Exactitud, Consistencia)
5. Generacion de Eventos de Seguridad
6. Deteccion de Anomalias
7. Simulacion 18,000 vs <100
8. Visualizaciones
9. Conclusiones Criticas

---
## 1. Marco DAMA DMBOK

| Metrica | Definicion | Formula | Umbral |
|---------|-----------|---------|--------|
| **Completitud** | Datos completos (sin nulos ni vacios) | `(no_nulos / total) x 100` | >= 95% |
| **Exactitud** | Valores dentro de rangos validos | `(validos / total) x 100` | >= 90% |
| **Consistencia** | Coherencia entre tablas y relaciones | `(rel_validas / total) x 100` | = 100% |

---
## 2. Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime, timedelta

np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (10, 6), 'font.size': 11,
    'axes.titlesize': 14, 'axes.labelsize': 12,
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.3,
})

COLORES = {'primario': '#1a73e8', 'secundario': '#ea4335', 'exito': '#34a853',
           'alerta': '#fbbc04', 'critico': '#d93025'}

print('Librerias cargadas')

In [ ]:
clientes = pd.read_csv('../data/clientes.csv')
versiones = pd.read_csv('../data/versiones_software.csv')
instalaciones = pd.read_csv('../data/instalaciones.csv')

print('Datos cargados:')
print(f'  Clientes: {clientes.shape}')
print(f'  Versiones: {versiones.shape}')
print(f'  Instalaciones: {instalaciones.shape}')

# Primera inspeccion: los clientes tienen 51 filas, no 50
# Esto ya es una anomalia (deberian ser 50)
print(f'\nALERTA: Se esperaban 50 clientes pero hay {len(clientes)}')

---
## 3. Validacion Exhaustiva

### 3.1 Valores Nulos

In [ ]:
print('=== VALORES NULOS ===')
for nombre, df in [('clientes', clientes), ('versiones', versiones), ('instalaciones', instalaciones)]:
    nulos = df.isnull().sum()
    nulos_con = nulos[nulos > 0]
    if len(nulos_con) > 0:
        print(f'\n  PROBLEMAS en {nombre}:')
        for col, n in nulos_con.items():
            filas = df[df[col].isnull()].index.tolist()
            print(f'    {col}: {n} nulos (filas: {filas})')
    else:
        print(f'\n  OK: {nombre} sin nulos')

### 3.2 IDs Duplicados

In [ ]:
print('=== IDs DUPLICADOS ===')

dups_cli = clientes[clientes['cliente_id'].duplicated(keep=False)]
if len(dups_cli) > 0:
    print(f'\n  CRITICO: {len(dups_cli)} filas con cliente_id duplicado:')
    print(dups_cli[['cliente_id', 'nombre_organizacion', 'tipo_org']].to_string())
    print('\n  IMPACTO: Imposible determinar cual es el registro correcto.')
    print('  ACCION: Requiere revision manual y deduplicacion.')
else:
    print('  OK: Sin IDs duplicados en clientes')

print(f'\n  version_id duplicados: {versiones["version_id"].duplicated().sum()}')
print(f'  instalacion_id duplicados: {instalaciones["instalacion_id"].duplicated().sum()}')

### 3.3 Integridad Referencial

In [ ]:
print('=== INTEGRIDAD REFERENCIAL ===')

cli_ids = set(clientes['cliente_id'])
ver_ids = set(versiones['version_id'])

# FK cliente_id
fk_cli_inv = instalaciones[~instalaciones['cliente_id'].isin(cli_ids)]
if len(fk_cli_inv) > 0:
    print(f'\n  CRITICO: {len(fk_cli_inv)} FK cliente_id huerfanas:')
    for _, row in fk_cli_inv.iterrows():
        print(f'    Instalacion {int(row["instalacion_id"])}: cliente_id={int(row["cliente_id"])} NO EXISTE')
    print('  IMPACTO: Estas instalaciones no pueden vincularse a ningun cliente.')
else:
    print('  OK: Todas las FK cliente_id son validas')

# FK version_id
fk_ver_inv = instalaciones[~instalaciones['version_id'].isin(ver_ids)]
if len(fk_ver_inv) > 0:
    print(f'\n  CRITICO: {len(fk_ver_inv)} FK version_id huerfanas:')
    for _, row in fk_ver_inv.iterrows():
        print(f'    Instalacion {int(row["instalacion_id"])}: version_id={int(row["version_id"])} NO EXISTE')
    print('  IMPACTO: No se puede determinar si esta instalacion tiene SUNBURST.')

### 3.4 Consistencia Temporal y Fechas

In [ ]:
print('=== CONSISTENCIA TEMPORAL ===')

# Fechas fuera de rango
fechas = pd.to_datetime(instalaciones['fecha_instalacion'], errors='coerce')
fuera = fechas[(fechas < '2019-01-01') | (fechas > '2021-06-30')]
if len(fuera) > 0:
    print(f'\n  CRITICO: {len(fuera)} fechas fuera del rango 2019-2021:')
    for idx in fuera.index:
        print(f'    Fila {idx+1}: {instalaciones.loc[idx, "fecha_instalacion"]}')

# Instalaciones antes del release
ver_con_fecha = versiones.dropna(subset=['fecha_release'])
merged = instalaciones.merge(ver_con_fecha[['version_id', 'fecha_release']], on='version_id', how='inner')
merged['f_inst'] = pd.to_datetime(merged['fecha_instalacion'])
merged['f_rel'] = pd.to_datetime(merged['fecha_release'])
incons = merged[merged['f_inst'] < merged['f_rel']]

if len(incons) > 0:
    print(f'\n  CRITICO: {len(incons)} instalaciones ANTES del release:')
    for _, row in incons.iterrows():
        print(f'    Instalacion {int(row["instalacion_id"])}: '
              f'instalado {row["fecha_instalacion"]} < release {row["fecha_release"]}')
    print('  IMPACTO: Imposible instalar software antes de que exista.')

# Versiones sin fecha
ver_sin = versiones[versiones['fecha_release'].isnull()]
if len(ver_sin) > 0:
    print(f'\n  ADVERTENCIA: {len(ver_sin)} versiones sin fecha_release:')
    for _, row in ver_sin.iterrows():
        print(f'    Version {int(row["version_id"])}: {row["nombre_version"]}')

### 3.5 Valores Categoricos Invalidos

In [ ]:
print('=== VALORES CATEGORICOS ===')

TIPOS_ORG_VALIDOS = ['Gobierno Federal', 'Gobierno Estatal', 'Empresa Privada',
                     'Institucion Educativa', 'Organizacion de Salud', 'ONG']

for col, validos in [('tipo_org', TIPOS_ORG_VALIDOS),
                      ('sector', ['Tecnologia', 'Defensa', 'Energia', 'Finanzas',
                                  'Telecomunicaciones', 'Salud', 'Gobierno', 'Educacion']),
                      ('criticidad', ['Alta', 'Media', 'Baja'])]:
    inv = clientes[clientes[col].notna() & ~clientes[col].isin(validos)]
    if len(inv) > 0:
        print(f'\n  PROBLEMA en clientes.{col}:')
        print(f'    Valores invalidos: {inv[col].unique().tolist()}')
        print(f'    Filas afectadas: {inv.index.tolist()}')
    else:
        print(f'  OK: clientes.{col}')

# nivel_datos_sensibles
inv_nivel = instalaciones[
    instalaciones['nivel_datos_sensibles'].notna() &
    ~instalaciones['nivel_datos_sensibles'].isin(['Bajo', 'Medio', 'Alto', 'Critico'])
]
if len(inv_nivel) > 0:
    print(f'\n  PROBLEMA en instalaciones.nivel_datos_sensibles:')
    print(f'    Valores invalidos: {inv_nivel["nivel_datos_sensibles"].unique().tolist()}')

---
## 4. Metricas DAMA DMBOK

Ahora calculamos las metricas formales. A diferencia de un analisis superficial, estas metricas **reflejan los problemas reales** encontrados.

In [ ]:
import sys
sys.path.insert(0, '../scripts')
from rol2_calidad_datos import calcular_metricas_dama

reporte_calidad = calcular_metricas_dama(clientes, versiones, instalaciones)

In [ ]:
# Mostrar metricas FALLIDAS (las que importan)
print('\n=== METRICAS FALLIDAS ===')
fallidas = reporte_calidad[reporte_calidad['estado'] == 'FALLIDO']
if len(fallidas) > 0:
    for _, row in fallidas.iterrows():
        deficit = row['umbral'] - row['valor_actual']
        print(f'  {row["tabla"]}.{row["columna"]}')
        print(f'    [{row["metrica"]}] {row["valor_actual"]}% (umbral: {row["umbral"]}%, deficit: -{deficit:.2f}%)')
        print()
else:
    print('  Todas las metricas aprobadas')

print(f'\nResumen: {len(fallidas)} de {len(reporte_calidad)} metricas FALLIDAS')

In [ ]:
# Tabla completa de metricas
reporte_calidad

---
## 5. Generacion de Eventos de Seguridad

In [ ]:
from rol2_calidad_datos import generar_eventos_seguridad

eventos_df = generar_eventos_seguridad(instalaciones, versiones, n=200)

print(f'\nDistribucion de tipos de evento:')
print(eventos_df['tipo_evento'].value_counts().to_string())
print(f'\nDistribucion de severidad:')
print(eventos_df['severidad'].value_counts().to_string())

In [ ]:
# Eventos anomalos: detalle
anomalos = eventos_df[eventos_df['es_anomalo'] == True]
print(f'Eventos anomalos: {len(anomalos)} de {len(eventos_df)} ({round(len(anomalos)/len(eventos_df)*100,1)}%)')
print(f'\nDetalle de eventos anomalos:')
anomalos

---
## 6. Deteccion de Anomalias

In [ ]:
from rol2_calidad_datos import detectar_anomalias

anomalias = detectar_anomalias(instalaciones, eventos_df, versiones, clientes)

---
## 7. Simulacion: 18,000 vs <100

In [ ]:
print('=== CASCADA DE FILTRADO: 18,000 -> <100 ===')
print()

versiones_sunburst = versiones[versiones['contiene_sunburst'] == True]['version_id'].tolist()

# Paso 1
total = len(instalaciones)
print(f'1. Total instalaciones: {total}')

# Paso 2
inst_sunburst = instalaciones[instalaciones['version_id'].isin(versiones_sunburst)]
print(f'2. Con versiones SUNBURST: {len(inst_sunburst)} ({round(len(inst_sunburst)/total*100,1)}%)')

# Paso 3: con eventos
ids_eventos = set(eventos_df['instalacion_id'])
con_eventos = inst_sunburst[inst_sunburst['instalacion_id'].isin(ids_eventos)]
print(f'3. Con eventos registrados: {len(con_eventos)}')

# Paso 4: anomalos
ids_anom = set(anomalos['instalacion_id'])
comprometidos = inst_sunburst[inst_sunburst['instalacion_id'].isin(ids_anom)]
print(f'4. Realmente comprometidos: {len(comprometidos)}')

# Clientes unicos (filtrando FK invalidos)
comp_validos = comprometidos[comprometidos['cliente_id'].isin(set(clientes['cliente_id']))]
clientes_comp = comp_validos['cliente_id'].nunique()
print(f'5. Clientes unicos comprometidos: {clientes_comp}')

print(f'\n--- Interpretacion ---')
print(f'Caso real: 18,000 -> <100 (reduccion >99.4%)')
print(f'Simulacion: {len(inst_sunburst)} -> {len(comprometidos)} '
      f'(reduccion {round((1-len(comprometidos)/max(len(inst_sunburst),1))*100,1)}%)')
print(f'\nLa diferencia entre "afectados" y "comprometidos" es la clave del caso.')
print(f'SolarWinds reporto 18,000 para ser transparente, pero el impacto real fue <100.')

---
## 8. Visualizaciones

In [ ]:
# Grafico 1: Anomalias
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

anom_c = eventos_df['es_anomalo'].value_counts()
axes[0].pie([anom_c.get(False, 0), anom_c.get(True, 0)],
            labels=['Normal', 'Anomalo'],
            colors=[COLORES['primario'], COLORES['critico']],
            explode=(0, 0.1), autopct='%1.1f%%', shadow=True, startangle=90,
            textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[0].set_title('Eventos Anomalos vs Normales', fontweight='bold')

if len(anomalos) > 0:
    tc = anomalos['tipo_evento'].value_counts()
    axes[1].barh(tc.index, tc.values, color=COLORES['critico'], alpha=0.8)
    axes[1].set_xlabel('Cantidad')
    axes[1].set_title('Tipos de Eventos Anomalos', fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/anomalias.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Grafico 2: Severidad
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sev = ['Baja', 'Media', 'Alta', 'Critica']
cols = ['#34a853', '#fbbc04', '#ea4335', '#d93025']

for val, label, ax in [(False, 'Normales', axes[0]), (True, 'Anomalos', axes[1])]:
    sub = eventos_df[eventos_df['es_anomalo'] == val]
    if len(sub) > 0:
        sc = sub['severidad'].value_counts().reindex(sev, fill_value=0)
        ax.bar(sc.index, sc.values, color=cols, edgecolor='white', linewidth=1.5)
        ax.set_title(f'{label}', fontweight='bold')
        ax.set_ylabel('Cantidad')

plt.suptitle('Distribucion de Severidad', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visualizations/severidad_eventos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Grafico 3: Clientes
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

crit = clientes['criticidad'].value_counts().reindex(['Alta', 'Media', 'Baja'], fill_value=0)
n_nulos = clientes['criticidad'].isnull().sum()
labels = list(crit.index) + ([f'NULL ({n_nulos})'] if n_nulos > 0 else [])
vals = list(crit.values) + ([n_nulos] if n_nulos > 0 else [])
colors = [COLORES['critico'], COLORES['alerta'], COLORES['exito']]
if n_nulos > 0: colors.append('#999999')

axes[0].bar(labels, vals, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Criticidad de Clientes', fontweight='bold')
for i, v in enumerate(vals):
    axes[0].text(i, v + 0.3, str(v), ha='center', fontweight='bold', fontsize=13)

sc = clientes['sector'].value_counts()
axes[1].barh(sc.index, sc.values, color=sns.color_palette('Blues_d', len(sc)))
axes[1].set_title('Sectores', fontweight='bold')

plt.suptitle('Analisis de Clientes', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visualizations/clientes_afectados.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Grafico 4: Heatmap de calidad
fig, ax = plt.subplots(figsize=(14, 10))

pivot = reporte_calidad.pivot_table(index=['tabla', 'columna'], columns='metrica',
                                    values='valor_actual', aggfunc='first')
labels = [f'{t} | {c}' for t, c in pivot.index]

sns.heatmap(pivot.values, xticklabels=pivot.columns, yticklabels=labels,
            annot=True, fmt='.1f', cmap='RdYlGn', vmin=85, vmax=100,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Porcentaje (%)'}, ax=ax)
ax.set_title('Metricas DAMA DMBOK - Analisis Critico', fontweight='bold', fontsize=14)

n_fail = len(reporte_calidad[reporte_calidad['estado'] == 'FALLIDO'])
ax.text(0.5, -0.06, f'{n_fail} metricas FALLIDAS detectadas',
        transform=ax.transAxes, ha='center', fontsize=12,
        color=COLORES['critico'], fontweight='bold')

plt.tight_layout()
plt.savefig('../visualizations/calidad_datos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Grafico 5: Cascada
fig, ax = plt.subplots(figsize=(12, 7))

cascada = anomalias.get('cascada', {})
etapas = ['Total\nDescargas\n(Real)', 'SUNBURST\n(Real)', 'Instalaciones\nSimuladas',
          'Con SUNBURST', 'Comprometidos', 'Clientes\nUnicos']
valores = [18000, 18000, cascada.get('total_instalaciones', 100),
           cascada.get('con_sunburst', 77), cascada.get('comprometidos', 11),
           cascada.get('clientes_unicos', 10)]

bars = ax.bar(range(len(etapas)), valores,
              color=['#1a73e8', '#4285f4', '#5f6368', '#ea4335', '#d93025', '#b71c1c'],
              edgecolor='white', linewidth=2, width=0.6)

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=13)

ax.set_xticks(range(len(etapas)))
ax.set_xticklabels(etapas, fontsize=9)
ax.set_title('Cascada: 18,000 -> <100 Comprometidos', fontweight='bold', fontsize=14)
ax.axvline(x=1.5, color='gray', linestyle='--', alpha=0.5)
ax.set_ylim(0, max(valores) * 1.15)

plt.tight_layout()
plt.savefig('../visualizations/grafico_cascada.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Exportacion de Datos

In [ ]:
eventos_df.to_csv('../data/eventos_seguridad.csv', index=False, encoding='utf-8')
reporte_calidad.to_csv('../data/reporte_calidad.csv', index=False, encoding='utf-8')

print('CSVs del Rol 2 guardados:')
print(f'  eventos_seguridad.csv: {len(eventos_df)} registros')
print(f'  reporte_calidad.csv: {len(reporte_calidad)} registros')

---
## 10. Conclusiones Criticas

### Problemas de Calidad Detectados

El analisis revelo **problemas reales** en los datos que impactan la confiabilidad del analisis:

| Categoria | Problema | Impacto | Severidad |
|-----------|---------|---------|----------|
| **Completitud** | 3 nombres de org. nulos, 1 criticidad nula, 2 niveles nulos | Registros incompletos no pueden analizarse | ALTO |
| **Completitud** | 1 fecha_release nula en versiones | No se puede verificar consistencia temporal | ALTO |
| **Exactitud** | Valores 'Desconocido' en tipo_org y nivel_sensibles | Categorias fuera de catalogo | MEDIO |
| **Exactitud** | 2 paises vacios (strings vacios, no nulos) | Datos presentes pero inutiles | MEDIO |
| **Consistencia** | 2 FK cliente_id huerfanas (999, 888) | Instalaciones sin cliente asociado | CRITICO |
| **Consistencia** | 1 FK version_id huerfana (99) | No se sabe si tiene SUNBURST | CRITICO |
| **Consistencia** | 3 fechas anteriores al release | Imposible temporalmente | CRITICO |
| **Consistencia** | 1 fecha en 2023 (fuera de rango) | Dato del futuro | ALTO |
| **Consistencia** | 1 cliente_id duplicado (ID=12) | Ambiguedad en la identidad | CRITICO |

### Metricas DAMA: 8 de 39 FALLIDAS (20.5%)

Esto significa que el **79.5%** de las metricas pasan, pero el 20.5% restante incluye problemas criticos de integridad referencial y consistencia temporal que **invalidan parcialmente** el analisis de seguridad.

### Leccion para la Gestion de Datos

1. **Un analisis que reporta 100% en todo no es creible**. Los datos reales siempre tienen problemas.
2. **Las FK huerfanas son el problema mas grave**: si no podemos vincular una instalacion a su cliente, no podemos evaluar el impacto del ataque.
3. **La consistencia temporal es critica en ciberseguridad**: una fecha incorrecta puede hacer que un evento parezca anterior al ataque cuando en realidad fue posterior.
4. **Los campos 'Desconocido' son peores que NULL**: un NULL es honesto (no hay dato), pero 'Desconocido' parece un dato real cuando no lo es.